In [1]:
!pip install transformers==4.50.0 datasets==3.5.0 accelerate==1.6.0 peft==0.15.0 bitsandbytes==0.45.2 -qqq

In [2]:
!pip uninstall bitsandbytes -y
!pip install bitsandbytes --no-cache-dir

Found existing installation: bitsandbytes 0.45.2
Uninstalling bitsandbytes-0.45.2:
  Successfully uninstalled bitsandbytes-0.45.2
   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   - -------------------------------------- 2.4/55.4 MB 12.3 MB/s eta 0:00:05
   --- ------------------------------------ 5.0/55.4 MB 11.6 MB/s eta 0:00:05
   ----- ---------------------------------- 7.3/55.4 MB 11.9 MB/s eta 0:00:05
   ------- -------------------------------- 9.7/55.4 MB 11.9 MB/s eta 0:00:04
   -------- ------------------------------- 12.1/55.4 MB 11.8 MB/s eta 0:00:04
   ---------- ----------------------------- 14.2/55.4 MB 11.7 MB/s eta 0:00:04
   ------------ --------------------------- 16.8/55.4 MB 11.6 MB/s eta 0:00:04
   ------------- -------------------------- 19.1/55.4 MB 11.6 MB/s eta 0:00:04
   --------------- ------------------------ 21.5/55.4 MB 11.6 MB/s eta 0:00:03
   ----------------- ---------------------- 24.1/55.4 MB 11.7 MB/s eta 0:00:03
   -----------

In [3]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import warnings
warnings.filterwarnings('ignore')

c:\Users\FORYOUCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 메모리 사용량 측정을 위한 함수 구현

In [6]:
import torch

def print_gpu_utilization():
    if torch.cuda.is_available():
        used_memory = torch.cuda.memory_allocated() / 1024**3
        print(f"GPU 메모리 사용량: {used_memory:.3f} GB")
    else:
        print("런타임 유형을 GPU로 변경하세요")

print_gpu_utilization()
# 출력 결과
# GPU 메모리 사용량: 0.000 GB

GPU 메모리 사용량: 0.000 GB


##  모델을 불러오고 GPU 메모리와 데이터 타입 확인

```python
load_model_and_tokenizer()
```
사용할 모델과 토크나이저의 아이디와 효율적인 모델 학습을 사용할지와 어떤 방식을 사용할지 입력받는 peft 인자를 받음

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tokenizer(model_id, peft=None):
    tokenizer = AutoTokenizer.from_pretrained(model_id) # 대응하는 모델과 토크나이저를 내려 받음
    if peft is None:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map={"":0})

    print_gpu_utilization()
    return model, tokenizer

model_id = "EleutherAI/polyglot-ko-1.3b"
model, tokenizer = load_model_and_tokenizer(model_id) # 모델과 토크나이저를 불러옴
 # GPU 메모리 사용량: 2.481 GB
print("모델 파라미터 데이터 타입: ", model.dtype) # torch.float16

Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.18it/s]


GPU 메모리 사용량: 2.481 GB
모델 파라미터 데이터 타입:  torch.float16


## 그레이디언트와 옵티마이저 상태의 메모리 사용량을 계산하는 함수

```python
estimate_memory_of_gradients()
```
인자로 모델을 받아 모델에 저장된 그레이디언트 값의 수와 값의 데이터 크기를 통해 전체 메모리 사용량 계산

```python
estimate_memory_of_optimizer()
```
인자로 옵티마이저를 받아 옵티마이저에 저장된 값의 수(v.nelement)와 값의 데이터 크기(v.element_size)를 곱해 전체 메모리 사용량 계산

In [9]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

def estimate_memory_of_gradients(model): # 그레이디언트의 메모리 사용량 확인
    total_memory = 0
    for param in model.parameters():
        if param.grad is not None:
            total_memory += param.grad.nelement() * param.grad.element_size()
    return total_memory

def estimate_memory_of_optimizer(optimizer): # 옵티마이저 상태의 메모리 사용량 확인
    total_memory = 0
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                total_memory += v.nelement() * v.element_size()
    return total_memory

## 모델의 학습 과정에서 메모리 사용량을 확인하는 train_model 정의

In [13]:
def train_model(model, dataset, training_args):
    if training_args.gradient_checkpointing:
        model.gradient_checkpointing_enable()

    train_dataloader = DataLoader(dataset, batch_size=training_args.per_device_train_batch_size)
    optimizer = AdamW(model.parameters())
    model.train()
    gpu_utilization_printed = False
    for step, batch in enumerate(train_dataloader, start=1):
        batch = {k: v.to(model.device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss = loss / training_args.gradient_accumulation_steps
        loss.backward()

        if step % training_args.gradient_accumulation_steps == 0:
            optimizer.step()
            gradients_memory = estimate_memory_of_gradients(model)
            optimizer_memory = estimate_memory_of_optimizer(optimizer)
            if not gpu_utilization_printed:
                print_gpu_utilization()
                gpu_utilization_printed = True
            optimizer.zero_grad()

    print(f"옵티마이저 상태의 메모리 사용량: {optimizer_memory / (1024 ** 3):.3f} GB")
    print(f"그레디언트 메모리 사용량: {gradients_memory / (1024 ** 3):.3f} GB")

## 랜덤 데이터셋을 생성하는 make_dummy_dataset 정의

In [14]:
import numpy as np
from datasets import Dataset

def make_dummy_dataset():
  seq_len, dataset_size = 256, 64
  dummy_data = {
      "input_ids": np.random.randint(100, 30000, (dataset_size, seq_len)),
      "labels": np.random.randint(100, 30000, (dataset_size, seq_len)),
  }
  dataset = Dataset.from_dict(dummy_data)
  dataset.set_format("pt")
  return dataset

## 더이상 사용하지 않는 GPU 메모리를 반환하는 cleanup 함수

In [15]:
import gc

def cleanup():
    if 'model' in globals():
        del globals()['model']
    if 'dataset' in globals():
        del globals()['dataset']
    gc.collect()
    torch.cuda.empty_cache()

## GPU 사용량을 확인하는 gpu_memory_experiment 함수 정의

In [16]:
from transformers import TrainingArguments, Trainer

def gpu_memory_experiment(batch_size,
                          gradient_accumulation_steps=1,
                          gradient_checkpointing=False,
                          model_id="EleutherAI/polyglot-ko-1.3b",
                          peft=None):

    print(f"배치 사이즈: {batch_size}")
    model, tokenizer = load_model_and_tokenizer(model_id, peft=peft)
    if gradient_checkpointing == True or peft == 'qlora':
        model.config.use_cache = False

    dataset = make_dummy_dataset()

    training_args = TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        gradient_checkpointing=gradient_checkpointing,
        output_dir="./result",
        num_train_epochs=1
      )

    try:
        train_model(model, dataset, training_args)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(e)
        else:
            raise e
    finally:
        del model, dataset
        gc.collect()
        torch.cuda.empty_cache()
        print_gpu_utilization()

## 배치 사이즈를 변경하며 메모리 사용량 측정

In [17]:
cleanup()
print_gpu_utilization()

for batch_size in [4, 8, 16]:
    gpu_memory_experiment(batch_size)

    torch.cuda.empty_cache()

GPU 메모리 사용량: 0.000 GB
배치 사이즈: 4


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.19it/s]


GPU 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 10.375 GB
옵티마이저 상태의 메모리 사용량: 4.961 GB
그레디언트 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 0.017 GB
배치 사이즈: 8


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.54it/s]


GPU 메모리 사용량: 2.498 GB
GPU 메모리 사용량: 10.808 GB
옵티마이저 상태의 메모리 사용량: 4.961 GB
그레디언트 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 0.017 GB
배치 사이즈: 16


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.18it/s]


GPU 메모리 사용량: 2.498 GB
GPU 메모리 사용량: 11.672 GB
옵티마이저 상태의 메모리 사용량: 4.961 GB
그레디언트 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 0.017 GB


## 그레이디언트 누적을 적용했을 때 메모리 사용량

In [18]:
cleanup()
print_gpu_utilization()

gpu_memory_experiment(batch_size=4, gradient_accumulation_steps=4)

torch.cuda.empty_cache()

GPU 메모리 사용량: 0.017 GB
배치 사이즈: 4


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.99it/s]


GPU 메모리 사용량: 2.498 GB
GPU 메모리 사용량: 10.375 GB
옵티마이저 상태의 메모리 사용량: 4.961 GB
그레디언트 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 0.017 GB


## 그레이디언트 체크포인팅 사용 시 메모리 사용량

In [19]:
cleanup()
print_gpu_utilization()

gpu_memory_experiment(batch_size=16, gradient_checkpointing=True)

torch.cuda.empty_cache()

GPU 메모리 사용량: 0.017 GB
배치 사이즈: 16


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.18it/s]


GPU 메모리 사용량: 2.498 GB
GPU 메모리 사용량: 10.174 GB
옵티마이저 상태의 메모리 사용량: 4.961 GB
그레디언트 메모리 사용량: 2.481 GB
GPU 메모리 사용량: 0.018 GB


## 모델을 불러오면서 LoRA 적용하기

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

def load_model_and_tokenizer(model_id, peft=None):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if peft is None:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map={"":0})

    elif peft == 'lora':
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map={"":0})
        lora_config = LoraConfig(
                    r=8,
                    lora_alpha=32,
                    target_modules=["query_key_value"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM"
                )

        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    print_gpu_utilization()
    return model, tokenizer

## LoRA를 적용했을 때 GPU 메모리 사용량 확인

In [21]:
cleanup()
print_gpu_utilization()

gpu_memory_experiment(batch_size=16, peft='lora')

torch.cuda.empty_cache()

GPU 메모리 사용량: 0.018 GB
배치 사이즈: 16


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.20it/s]


trainable params: 1,572,864 || all params: 1,333,383,168 || trainable%: 0.1180
GPU 메모리 사용량: 2.504 GB
GPU 메모리 사용량: 4.252 GB
옵티마이저 상태의 메모리 사용량: 0.012 GB
그레디언트 메모리 사용량: 0.006 GB
GPU 메모리 사용량: 0.018 GB


## 4비트 양자화 모델 불러오기

In [22]:
from transformers import BitsAndBytesConfig
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model_nf4 = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=nf4_config)

`low_cpu_mem_usage` was None, now default to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.35it/s]


## QLoRA 모델을 불러오는 부분을 추가

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def load_model_and_tokenizer(model_id, peft=None):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if peft is None:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map={"":0})

    elif peft == 'lora':
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map={"":0})
        lora_config = LoraConfig(
                    r=8,
                    lora_alpha=32,
                    target_modules=["query_key_value"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM"
                )

        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    elif peft == 'qlora':
        lora_config = LoraConfig(
                    r=8,
                    lora_alpha=32,
                    target_modules=["query_key_value"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM"
                )
        bnb_config = BitsAndBytesConfig(
                  load_in_4bit=True,
                  bnb_4bit_use_double_quant=True,
                  bnb_4bit_quant_type="nf4",
                  bnb_4bit_compute_dtype=torch.float16
              )
        model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
        model.gradient_checkpointing_enable()
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    print_gpu_utilization()
    return model, tokenizer

## QLoRA를 적용했을 때 GPU 메모리 사용량 확인

In [24]:
cleanup()
print_gpu_utilization()

gpu_memory_experiment(batch_size=16, peft='qlora')

torch.cuda.empty_cache()

GPU 메모리 사용량: 0.830 GB
배치 사이즈: 16


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.11it/s]


trainable params: 1,572,864 || all params: 1,333,383,168 || trainable%: 0.1180
GPU 메모리 사용량: 1.880 GB
GPU 메모리 사용량: 2.419 GB
옵티마이저 상태의 메모리 사용량: 0.012 GB
그레디언트 메모리 사용량: 0.006 GB
GPU 메모리 사용량: 0.830 GB
